[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_backtest_overfitting/MFM_ch13_backtest_overfitting.ipynb)

# MFM_ch13_backtest_overfitting

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Backtest overfitting: expected maximum Sharpe ratio of zero-skill strategies (False Strategy Theorem) versus Monte Carlo, in-sample versus out-of-sample Sharpe of a BTC moving-average grid search, and the Deflated Sharpe Ratio as a function of the number of trials.

*Author: Daniel Traian Pele*

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


In [ ]:
import os
DATA_URL = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
SERIES = {'sp500': 'GSPC.INDX', 'vix': 'VIX.INDX', 'btc': 'BTC-USD.CC'}
START = {'sp500': '2000-01-01', 'vix': '2000-01-01', 'btc': '2014-09-17'}


def load_data(name='sp500', start=None, end='2026-09-18'):
    """OHLCV zilnic din data/market: copia locala, altfel fisierul din repo-ul GitHub MFM."""
    fname = SERIES[name] + '.csv'
    local = [os.path.join(d, 'data', 'market', fname) for d in ('.', '..', '../..', '../../..')]
    path = next((p for p in local if os.path.exists(p)), DATA_URL + fname)
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df = df.rename(columns=str.capitalize)[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df.loc[start or START[name]:end]


spx = load_data('sp500')['Close']
vix = load_data('vix')['Close']
btc = load_data('btc')['Close']
log_spx = np.log(spx)
print('S&P 500:', spx.index[0].date(), '->', spx.index[-1].date(), len(spx), 'obs')
print('BTC    :', btc.index[0].date(), '->', btc.index[-1].date(), len(btc), 'obs')

## Helper functions

In [ ]:
EULER_GAMMA = 0.5772156649015329

def sharpe_ratio(returns, periods=252):
    """Sharpe anualizat (fara rata fara risc)."""
    r = np.asarray(returns)
    return np.sqrt(periods) * r.mean() / r.std(ddof=1)


def probabilistic_sharpe_ratio(sr, sr_benchmark, T, skew=0.0, kurt=3.0):
    """
    PSR(SR*) = Phi( (SR - SR*) sqrt(T-1) / sqrt(1 - g3 SR + (g4 - 1)/4 SR^2) )
    SR si SR* sunt NEanualizate (pe perioada de observatie); kurt = kurtosis (distributia Normala = 3).
    """
    denom = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr ** 2)
    return stats.norm.cdf((sr - sr_benchmark) * np.sqrt(T - 1) / denom)


def expected_max_sharpe(n_trials, sr_std=1.0):
    """
    Teorema strategiei false (Bailey & Lopez de Prado, 2014):
    E[max SR_n] ~ sqrt(V[SR]) * ((1-g) Phi^{-1}(1 - 1/N) + g Phi^{-1}(1 - 1/(N e)))
    """
    n = np.asarray(n_trials, dtype=float)
    return sr_std * ((1 - EULER_GAMMA) * stats.norm.ppf(1 - 1 / n)
                     + EULER_GAMMA * stats.norm.ppf(1 - 1 / (n * np.e)))


def deflated_sharpe_ratio(sr, T, n_trials, sr_std, skew=0.0, kurt=3.0):
    """DSR = PSR(SR_0), cu SR_0 = E[max SR] sub ipoteza nula de lipsa a abilitatii."""
    sr0 = expected_max_sharpe(n_trials, sr_std)
    return probabilistic_sharpe_ratio(sr, sr0, T, skew, kurt)

## Charts

In [ ]:
def fig_expected_max_sharpe():
    rng = np.random.default_rng(SEED)
    T = 252 * 5
    Ns = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000])
    sims = []
    for N in Ns:
        # N strategii fara abilitate: randamente iid N(0, 1%) pe 5 ani, 200 repetitii
        r = rng.normal(0, 0.01, size=(200, N, T))
        sr = np.sqrt(252) * r.mean(-1) / r.std(-1, ddof=1)
        sims.append(sr.max(1))
    sims = np.array(sims)
    grid = np.logspace(np.log10(2), 3, 200)
    theory = expected_max_sharpe(grid, sr_std=np.sqrt(252 / T))

    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    ax.plot(grid, theory, color=IDAred, lw=1.2, label='$E[\\max SR]$ (False Strategy Theorem)')
    ax.errorbar(Ns, sims.mean(1), yerr=sims.std(1), fmt='o', ms=3.5, color=MainBlue,
                capsize=2, lw=0.8, label='Monte Carlo (mean $\\pm$ 1 sd)')
    ax.set_xscale('log')
    ax.set_xlabel('Number of independent strategies tried $N$')
    ax.set_ylabel('Best annualised Sharpe ratio')
    ax.set_title('Zero-skill strategies, 5 years of daily data: the best one always "works"',
                 fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    plt.tight_layout()
    save_fig('ch13_expected_max_sharpe')

In [ ]:
def fig_is_vs_oos():
    """Cautare de parametri pentru o strategie de medii mobile pe BTC: IS 2015-2020, OOS 2021-2026."""
    r = np.log(btc).diff()
    lp = np.log(btc)
    is_mask = btc.index < '2021-01-01'
    rows = []
    for fast in range(2, 60, 3):
        for slow in range(10, 250, 10):
            if slow <= fast:
                continue
            for band in (0.0, 0.01, 0.03):
                sig = ((lp.rolling(fast).mean() - lp.rolling(slow).mean()) > band).astype(float).shift(1)
                sr_is = sharpe_ratio((sig * r)[is_mask].dropna(), 365)
                sr_oos = sharpe_ratio((sig * r)[~is_mask].dropna(), 365)
                rows.append((fast, slow, band, sr_is, sr_oos))
    res = pd.DataFrame(rows, columns=['fast', 'slow', 'band', 'sr_is', 'sr_oos']).dropna()
    best = res.loc[res['sr_is'].idxmax()]
    rank_corr = stats.spearmanr(res['sr_is'], res['sr_oos'])[0]
    top = res.nlargest(int(len(res) * 0.1), 'sr_is')

    fig, ax = plt.subplots(figsize=(6.4, 3.1))
    ax.scatter(res['sr_is'], res['sr_oos'], s=6, color=MainBlue, alpha=0.35, label=f'{len(res)} configurations')
    ax.scatter(top['sr_is'], top['sr_oos'], s=8, color=Amber, alpha=0.8, label='Top 10% in-sample')
    ax.scatter(best['sr_is'], best['sr_oos'], s=70, marker='*', color=IDAred, zorder=3,
               label=f"Best IS: SR$_{{IS}}$={best['sr_is']:.2f}, SR$_{{OOS}}$={best['sr_oos']:.2f}")
    lims = [min(res['sr_is'].min(), res['sr_oos'].min()) - 0.1, max(res['sr_is'].max(), res['sr_oos'].max()) + 0.1]
    ax.plot(lims, lims, color=Gray, ls=':', lw=0.8)
    ax.set_xlabel('Sharpe in-sample (2015-2020)')
    ax.set_ylabel('Sharpe out-of-sample (2021-2026)')
    ax.set_title(f'BTC moving-average crossover grid search (Spearman $\\rho$ = {rank_corr:.2f})',
                 fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    plt.tight_layout()
    save_fig('ch13_is_vs_oos')
    return res, best, rank_corr, top['sr_oos'].median()

In [ ]:
def fig_deflated_sharpe():
    T = 252 * 5
    N = np.unique(np.logspace(0, 3, 60).astype(int))
    sr_std = np.sqrt(1 / T)                       # dispersia SR (neanualizat) sub H0
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    for sr_ann, c in zip([1.0, 1.5, 2.0, 2.5], [IDAred, Amber, Forest, MainBlue]):
        sr = sr_ann / np.sqrt(252)
        dsr = deflated_sharpe_ratio(sr, T, np.maximum(N, 1.0001), sr_std, skew=-0.5, kurt=6.0)
        ax.plot(N, dsr, color=c, label=f'observed SR = {sr_ann}')
    ax.axhline(0.95, color=Gray, ls='--', lw=0.8)
    ax.text(1.05, 0.90, '95% confidence', fontsize=7.5, color='black')
    ax.set_xscale('log')
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('Number of trials $N$')
    ax.set_ylabel('Deflated Sharpe Ratio')
    ax.set_title('DSR: 5 years daily, skew = -0.5, kurtosis = 6', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=4, y=-0.25)
    plt.tight_layout()
    save_fig('ch13_deflated_sharpe')

## Run

In [ ]:
fig_expected_max_sharpe()
res, best, rho, top_oos = fig_is_vs_oos()
print(f'{len(res)} configurations; best IS: {best.to_dict()}; Spearman rho = {rho:.2f}')
fig_deflated_sharpe()

![ch13_expected_max_sharpe](./ch13_expected_max_sharpe.png)

![ch13_is_vs_oos](./ch13_is_vs_oos.png)

![ch13_deflated_sharpe](./ch13_deflated_sharpe.png)

Output: `ch13_expected_max_sharpe.pdf`, `ch13_is_vs_oos.pdf`, `ch13_deflated_sharpe.pdf`